In [4]:
import sys
sys.path.append("/Users/avelezxerenity/Documents/GitHub/pysdk")


from db_call.db_call import get_last_banrep,get_ibr_cluster_table
from loans_calculator.loan_structure import Loan
from datetime import datetime


from datetime import datetime
import pandas as pd
import QuantLib as ql
from swap_functions.main import full_ibr_curve_creation
from utilities.colombia_calendar import calendar_colombia


In [5]:
periodicity="Mensual"
interest_rate=9.53
periodicity="Mensual"
number_of_payments=12
datetime_date="2024-05-21"
start_date=datetime.strptime(datetime_date, '%Y-%m-%d')
original_balance=1000
rate_type='IBR'

SV=get_last_banrep("Indicador Bancario de Referencia (IBR) 6 Meses, nominal",365*5).data
initial_date='2024-05-13 00:00:00'
final_date='2024-05-24 19:17:34'

ibr_cluster_table=get_ibr_cluster_table(initial_date=initial_date,final_date=final_date)
TV=get_last_banrep("Indicador Bancario de Referencia (IBR) 3 Meses, nominal",365*5).data
MV=get_last_banrep("Indicador Bancario de Referencia (IBR) 1 Mes, nominal",365*5).data
ibr_1m=get_last_banrep("Indicador Bancario de Referencia (IBR) 1 Mes, nominal").data[0]['valor']

db_info={'SV': SV,
        'ibr_cluster_table': ibr_cluster_table,
        'TV': TV,
        'MV': MV,
        'ibr_1m': ibr_1m}

2024-05-24 18:17:07,129:INFO - HTTP Request: GET https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/banrep_series_value_v2?select=%2A&id_serie=eq.15&order=fecha.desc&limit=1825 "HTTP/1.1 200 OK"
2024-05-24 18:17:07,668:INFO - HTTP Request: GET https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/ibr_swaps_cluster?select=%2A "HTTP/1.1 200 OK"
/Users/avelezxerenity/Documents/GitHub/pysdk/src/xerenity/xty.py:141: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column] = pd.to_datetime(df[column])
/Users/avelezxerenity/Documents/GitHub/pysdk/src/xerenity/xty.py:141: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column] = pd.to_datetime(df[column])
2024-05-24 18:17:08,484:INFO - HTTP Request: GET https://tvpehjbqxpisw

In [6]:
calc=Loan(interest_rate=interest_rate,periodicity=periodicity,
          number_of_payments=number_of_payments,
          start_date=start_date,
          original_balance=original_balance,
          rate_type=rate_type,
          db_info=db_info)

In [7]:

calc.rate_type = 'IBR'
today = datetime.today().date()
start = ql.Date(today.day, today.month, today.year)
ql_today = ql.Date(today.day, today.month, today.year)
calendar = calendar_colombia()
depth_search = 8

while not calendar.isBusinessDay(start) and depth_search >= 0:
    start = calendar.advance(start, -1, ql.Days)
    depth_search = depth_search - 1

if depth_search == 0:
    print("No business day found in {} days".format(depth_search))
    start = ql_today

value_date = datetime(year=start.year(), month=start.month(), day=start.dayOfMonth())

curve_details = full_ibr_curve_creation(
    desired_date_valuation=ql.Date(value_date.day, value_date.month, value_date.year),
    calendar=calendar_colombia(),
    day_to_avoid_fwd_ois=7,
    db_info=calc.db_info
)




In [8]:
curve = curve_details.crear_curva(days_to_on=1)

payment = calc.generate_rates_ibr(
    value_date=value_date,
    curve=curve,
    tipo_de_cobro='por_dias_360',
    periodicidad_tasa='MV')

/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/loan_structure.py:189: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cf_table['date'] = cf_table['date'].apply(ql_to_datetime)


RuntimeError: 1st iteration: failed at 1st alive instrument, pillar June 28th, 2024, maturity June 28th, 2024, reference date May 24th, 2024: root not bracketed: f[0.907354,1.10211] -> [9.963682e+00,1.196615e+01]